# Lecture 10: Contact Forms

**Source span.** Printed pages 55-60; physical PDF pages 69-74 in `Lectures on Symplectic Geometry.pdf`. I inspected the local PDF text for this span before revising the notebook.

**Lecture goal.** Introduce contact elements, contact structures, contact forms, and the first normal-form and Gray-stability statements as visual and computable tests.

A contact structure is a hyperplane field that is as far from integrable as possible. If `H = ker alpha` locally on a `(2n+1)`-manifold, the test is that `d alpha` restricts to a symplectic form on `H`; equivalently, `alpha wedge (d alpha)^n` is a volume form. This notebook makes that test concrete in the standard `R3` model, compares it with an integrable plane field, and then connects the contact-element homework to `P(T*X)` and the cotangent sphere bundle.

In [ ]:
from pathlib import Path
import json
import sys

import matplotlib.pyplot as plt
import networkx as nx
import numpy as np


def find_book_root(start=None):
    start = Path.cwd() if start is None else Path(start)
    for candidate in [start, *start.parents]:
        if (candidate / "AGENTS.md").exists() and (candidate / "Lectures on Symplectic Geometry.pdf").exists():
            return candidate
    raise RuntimeError("Could not locate the Lectures-on-Symplectic-Geometry course root.")


BOOK_ROOT = find_book_root()
if str(BOOK_ROOT) not in sys.path:
    sys.path.insert(0, str(BOOK_ROOT))

from utils.artifacts import display_artifact, read_json, save_json, save_matplotlib

ARTIFACT_TOPIC = "lecture-10"
ARTIFACT_ROOT = BOOK_ROOT / "artifacts" / ARTIFACT_TOPIC
(ARTIFACT_ROOT / "figures").mkdir(parents=True, exist_ok=True)
(ARTIFACT_ROOT / "checks").mkdir(parents=True, exist_ok=True)
plt.rcParams.update({"figure.dpi": 130, "font.size": 10})
print(f"Artifact topic: {ARTIFACT_TOPIC}")

## Translation Guide And Library Routing

| Source idea | Computational object | Check |
| --- | --- | --- |
| contact element | a point plus a tangent hyperplane | hyperplane is `ker alpha_p` for a nonzero covector |
| scalar ambiguity | replace `alpha` by `f alpha` | kernel is unchanged and the contact volume scales by a nonzero factor |
| Contact structures | hyperplane field `H = ker alpha` | `d alpha | H` is nondegenerate |
| contact form | a local/global one-form defining `H` | `alpha wedge (d alpha)^n` is nowhere zero |
| contact distribution | basis vectors spanning `ker alpha` | bracket obstruction leaves the distribution |
| maximal nonintegrability | Frobenius failure in the strongest possible odd-dimensional way | `alpha([E1,E2])` is nonzero in the `R3` model |
| local normal form | coordinates with `alpha = sum x_i dy_i + dz` | the standard form has nonzero contact volume |
| Gray theorem | Moser-type equation on `H_t` | `i_v d alpha_t | H_t = - d/dt alpha_t | H_t` has a unique solution because `d alpha_t | H_t` is symplectic |

**Library routing.** `matplotlib` handles plane-field and contact-element diagrams, including a small 3D view. `numpy` computes volume coefficients, bracket tests, and sample scaling checks. `networkx` is used for the proof ledger because Gray's theorem is a sequence of dependent equation reductions rather than a single picture.

## Visualization Storyboard

1. **Contact planes twisting along a curve.** For `alpha = x dy + dz`, `ker alpha` is spanned by `partial_x` and `partial_y - x partial_z`. The planes tilt with `x`, while the integrable comparison `ker dz` stays horizontal.
2. **Frobenius failure meter.** The bracket `[partial_x, partial_y - x partial_z] = -partial_z` escapes the contact distribution; that is the visible meaning of maximal nonintegrability in dimension three.
3. **Contact elements and `P(T*X)`.** A hyperplane is the kernel of a covector up to nonzero scale. The local chart on projectivized cotangent space has contact form `dx1 + xi dx2`.
4. **Gray proof ledger.** The Moser-style proof solves a vector field inside `H_t`, integrates it, and records the conformal factor `u_t` because the contact structure is the kernel, not the particular one-form.

In [ ]:
# Standard R3 contact model alpha = x dy + dz and integrable comparison alpha0 = dz.
def contact_basis_at_x(x):
    return np.array([[1.0, 0.0, 0.0], [0.0, 1.0, -x]])

contact_volume_coeff = 1.0  # alpha wedge d alpha = dx wedge dy wedge dz for alpha = x dy + dz
integrable_volume_coeff = 0.0
bracket_escape = -1.0  # alpha(-partial_z) = -1
integrable_bracket_escape = 0.0
assert abs(contact_volume_coeff) > 0.99
assert integrable_volume_coeff == 0.0
assert abs(bracket_escape) > 0.99
assert integrable_bracket_escape == 0.0

xs = np.linspace(-1.2, 1.2, 7)
ys = np.zeros_like(xs)
zs = np.zeros_like(xs)
fig = plt.figure(figsize=(12, 5.2))
ax = fig.add_subplot(1, 2, 1, projection="3d")
for x, y, z in zip(xs, ys, zs):
    e1 = np.array([0.32, 0.0, 0.0])
    e2 = np.array([0.0, 0.32, -0.32 * x])
    ax.quiver(x, y, z, *e1, color="#1b9e77", linewidth=1.8)
    ax.quiver(x, y, z, *e2, color="#d95f02", linewidth=1.8)
    corners = np.array([[-1, -1], [1, -1], [1, 1], [-1, 1], [-1, -1]]) * 0.18
    patch = np.array([np.array([x, y, z]) + a * e1 + b * e2 for a, b in corners])
    ax.plot(patch[:, 0], patch[:, 1], patch[:, 2], color="0.45", alpha=0.7)
ax.set_title("contact planes: ker(x dy + dz)")
ax.set_xlabel("x")
ax.set_ylabel("y")
ax.set_zlabel("z")
ax.view_init(elev=23, azim=-58)

ax2 = fig.add_subplot(1, 2, 2, projection="3d")
for x, y, z in zip(xs, ys, zs):
    e1 = np.array([0.32, 0.0, 0.0])
    e2 = np.array([0.0, 0.32, 0.0])
    ax2.quiver(x, y, z, *e1, color="#1b9e77", linewidth=1.8)
    ax2.quiver(x, y, z, *e2, color="#2c7fb8", linewidth=1.8)
    corners = np.array([[-1, -1], [1, -1], [1, 1], [-1, 1], [-1, -1]]) * 0.18
    patch = np.array([np.array([x, y, z]) + a * e1 + b * e2 for a, b in corners])
    ax2.plot(patch[:, 0], patch[:, 1], patch[:, 2], color="0.45", alpha=0.7)
ax2.set_title("integrable comparison: ker(dz)")
ax2.set_xlabel("x")
ax2.set_ylabel("y")
ax2.set_zlabel("z")
ax2.view_init(elev=23, azim=-58)
fig.suptitle("Contact structures: the plane field twists enough to defeat integrability")
contact_planes_path = save_matplotlib(fig, ARTIFACT_TOPIC, "figures", "contact-planes-and-bracket-obstruction.png")
plt.close(fig)
display_artifact(contact_planes_path, width=760)
print({"contact_volume_coeff": contact_volume_coeff, "bracket_escape": bracket_escape})

In [ ]:
# Scaling alpha by a nowhere-zero function keeps ker(alpha), but rescales contact volume.
sample_x = np.linspace(-1.5, 1.5, 41)
sample_z = np.linspace(-1.0, 1.0, 31)
X, Z = np.meshgrid(sample_x, sample_z)
f = 1.0 + 0.2 * X**2 + 0.1 * Z**2
scaled_volume = f**2 * contact_volume_coeff
scaled_volume_min = float(np.min(scaled_volume))
scaled_volume_max = float(np.max(scaled_volume))
assert scaled_volume_min > 0.99

# Local contact chart on P(T*R2): alpha = dx1 + xi dx2, so alpha wedge d alpha is nonzero.
xi = np.linspace(-2.0, 2.0, 25)
base_x = np.zeros_like(xi)
base_y = np.zeros_like(xi)
line_dirs = np.vstack([-xi, np.ones_like(xi)]).T
line_dirs = line_dirs / np.linalg.norm(line_dirs, axis=1)[:, None]
projective_contact_volume_abs = 1.0
assert projective_contact_volume_abs > 0.99

fig, axes = plt.subplots(1, 2, figsize=(11.5, 4.4))
im = axes[0].imshow(scaled_volume, extent=[sample_x.min(), sample_x.max(), sample_z.min(), sample_z.max()], origin="lower", cmap="magma")
axes[0].set_title("(f alpha) wedge d(f alpha) = f^2 alpha wedge d alpha")
axes[0].set_xlabel("x")
axes[0].set_ylabel("z")
fig.colorbar(im, ax=axes[0], fraction=0.046)

chosen = np.array([-1.5, -0.5, 0.5, 1.5])
colors = ["#1b9e77", "#2c7fb8", "#d95f02", "#7570b3"]
for slope, color in zip(chosen, colors):
    direction = np.array([-slope, 1.0])
    direction = direction / np.linalg.norm(direction)
    normal = np.array([1.0, slope])
    normal = normal / np.linalg.norm(normal)
    t = np.linspace(-0.9, 0.9, 20)
    line = t[:, None] * direction
    axes[1].plot(line[:, 0], line[:, 1], color=color, lw=2, label=f"[1,{slope:+.1f}]")
    axes[1].arrow(0, 0, 0.35 * normal[0], 0.35 * normal[1], color=color, head_width=0.04, length_includes_head=True)
axes[1].set_aspect("equal", adjustable="box")
axes[1].axhline(0, color="0.85", lw=0.8)
axes[1].axvline(0, color="0.85", lw=0.8)
axes[1].set_title("contact elements: hyperplanes as covector kernels")
axes[1].set_xlabel("dx1 direction")
axes[1].set_ylabel("dx2 direction")
axes[1].legend(fontsize=8, title="covector class")
fig.suptitle("Projectivized cotangent chart: (x,[xi]) records a contact element")
projective_path = save_matplotlib(fig, ARTIFACT_TOPIC, "figures", "projectivized-cotangent-contact-elements.png")
plt.close(fig)
display_artifact(projective_path, width=760)
print({"scaled_volume_min": scaled_volume_min, "projective_contact_volume_abs": projective_contact_volume_abs})

In [ ]:
# Gray theorem proof ledger and a small family alpha_t = dz + (x + t*z) dy.
ts = np.linspace(0, 1, 21)
family_volume_coefficients = np.ones_like(ts)  # dz wedge dx wedge dy term stays 1 for this family
assert np.min(np.abs(family_volume_coefficients)) > 0.99

G = nx.DiGraph()
G.add_edges_from([
    ("smooth family of contact forms alpha_t", "hyperplanes H_t = ker alpha_t"),
    ("hyperplanes H_t = ker alpha_t", "seek isotopy rho_t"),
    ("seek isotopy rho_t", "differentiate rho_t^* alpha_t"),
    ("differentiate rho_t^* alpha_t", "Moser-style equation"),
    ("choose v_t inside H_t", "restrict equation to H_t"),
    ("restrict equation to H_t", "i_v d alpha_t | H_t = -partial_t alpha_t | H_t"),
    ("d alpha_t | H_t nondegenerate", "unique v_t"),
    ("unique v_t", "integrate to rho_t"),
    ("integrate to rho_t", "rho_t^* alpha_t = u_t alpha_0"),
    ("rho_t^* alpha_t = u_t alpha_0", "rho_t carries H_0 to H_t"),
])
pos = {
    "smooth family of contact forms alpha_t": (0, 2.0),
    "hyperplanes H_t = ker alpha_t": (1.8, 2.0),
    "seek isotopy rho_t": (3.6, 2.0),
    "differentiate rho_t^* alpha_t": (5.4, 2.0),
    "Moser-style equation": (7.2, 2.0),
    "choose v_t inside H_t": (1.8, 0.7),
    "restrict equation to H_t": (3.6, 0.7),
    "i_v d alpha_t | H_t = -partial_t alpha_t | H_t": (5.4, 0.7),
    "d alpha_t | H_t nondegenerate": (3.6, -0.6),
    "unique v_t": (5.4, -0.6),
    "integrate to rho_t": (7.2, -0.6),
    "rho_t^* alpha_t = u_t alpha_0": (7.2, 0.7),
    "rho_t carries H_0 to H_t": (9.0, 0.7),
}
fig, ax = plt.subplots(figsize=(13, 5.5))
nx.draw_networkx_edges(G, pos, ax=ax, arrows=True, arrowstyle="-|>", arrowsize=15, edge_color="0.35")
nx.draw_networkx_nodes(G, pos, ax=ax, node_color="#b3cde3", node_size=2200, edgecolors="0.25", linewidths=0.8)
nx.draw_networkx_labels(G, pos, ax=ax, font_size=8)
ax.text(0.1, -1.45, "Family check: alpha_t = dz + (x + t z) dy has alpha_t wedge d alpha_t coefficient 1 for every sampled t.", fontsize=10, bbox={"boxstyle": "round,pad=0.35", "fc": "white", "ec": "0.55"})
ax.set_axis_off()
ax.set_title("Gray theorem: contact Moser argument for moving hyperplane fields")
gray_path = save_matplotlib(fig, ARTIFACT_TOPIC, "figures", "gray-theorem-proof-ledger.png")
plt.close(fig)
display_artifact(gray_path, width=760)
print({"gray_family_min_volume": float(np.min(family_volume_coefficients))})

## Reading The Visuals

The twisting-plane figure is the standard example from the source span. At `x=0`, the contact plane looks horizontal, but as `x` changes the second basis vector picks up vertical velocity. The bracket of the two displayed spanning fields is `-partial_z`; applying `alpha` to that bracket gives a nonzero value, so the bracket cannot remain inside `H`. That is the small-dimensional Frobenius failure meter.

The projectivized-cotangent figure explains the contact-element homework without reproducing the exercise text. A nonzero covector and any nonzero scalar multiple have the same kernel, so a hyperplane is naturally a projective covector. In the local chart where a covector class is represented by `(1, xi)`, the form `dx1 + xi dx2` is contact because its contact volume coefficient is nonzero.

Gray's theorem is the contact analogue of Moser stability. The proof solves only on `H_t` because `d alpha_t | H_t` is nondegenerate there. The extra scalar `u_t` appears because contact structures remember kernels of forms, so multiplying the form by a nowhere-zero function is harmless.

In [ ]:
residuals = {
    "contact_volume_coeff_alpha_xdy_plus_dz": contact_volume_coeff,
    "integrable_volume_coeff_dz": integrable_volume_coeff,
    "contact_bracket_escape_alpha_value": bracket_escape,
    "integrable_bracket_escape": integrable_bracket_escape,
    "scaled_volume_min": scaled_volume_min,
    "scaled_volume_max": scaled_volume_max,
    "projectivized_cotangent_contact_volume_abs": projective_contact_volume_abs,
    "gray_family_min_volume_coeff": float(np.min(family_volume_coefficients)),
}
residual_path = save_json(residuals, ARTIFACT_TOPIC, "checks", "contact-form-residuals.json")

source_span = {
    "title": "Contact Forms",
    "printed_pages": "55-60",
    "physical_pdf_pages": "69-74",
    "source_file": "Lectures on Symplectic Geometry.pdf",
    "sections": [
        "Contact structures",
        "Examples",
        "First properties",
        "Gray theorem proof sketch",
        "Homework 7 contact elements themes",
    ],
    "concepts": [
        "contact form",
        "contact distribution",
        "maximal nonintegrability",
        "local normal form",
        "projectivized cotangent bundle",
        "cotangent sphere bundle",
    ],
    "copyright_note": "Source pages were used for terminology and coverage; notebook prose, visuals, and code are original and no page crops or long exercise text are copied.",
}
source_path = save_json(source_span, ARTIFACT_TOPIC, "checks", "source-span.json")

storyboard = {
    "chapter_goal": "Make the contact condition alpha wedge (d alpha)^n nonzero visible through twisting planes, bracket escape, and contact-element models.",
    "source_span_read": source_span,
    "library_routing": [
        {"concept": "contact distribution", "representation": "3D plane-field diagram", "library": "matplotlib", "why": "the source example asks the reader to picture hyperplanes"},
        {"concept": "maximal nonintegrability", "representation": "bracket escape and contact-volume residuals", "library": "numpy", "why": "the contact test is a scalar volume coefficient in dimension three"},
        {"concept": "contact elements", "representation": "projective covector kernels", "library": "matplotlib", "why": "projectivized cotangent space is easiest to inspect as kernel lines and covectors"},
        {"concept": "Gray theorem", "representation": "proof dependency graph", "library": "networkx", "why": "the proof is a chain of equation reductions inside H_t"},
    ],
    "visual_sequence": [
        {"concept": "contact planes twisting along a curve", "artifact": "artifacts/lecture-10/figures/contact-planes-and-bracket-obstruction.png", "inspection_target": "planes twist and the bracket exits the distribution"},
        {"concept": "contact elements and projectivized cotangent", "artifact": "artifacts/lecture-10/figures/projectivized-cotangent-contact-elements.png", "inspection_target": "covector classes encode tangent hyperplanes"},
        {"concept": "Gray theorem proof ledger", "artifact": "artifacts/lecture-10/figures/gray-theorem-proof-ledger.png", "inspection_target": "nondegeneracy on H_t gives the Moser vector field"},
    ],
    "computational_checks": residuals,
}
storyboard_path = save_json(storyboard, ARTIFACT_TOPIC, "checks", "visual-storyboard.json")

final_sanity = {
    "artifacts": [
        str(contact_planes_path.relative_to(BOOK_ROOT)),
        str(projective_path.relative_to(BOOK_ROOT)),
        str(gray_path.relative_to(BOOK_ROOT)),
        str(residual_path.relative_to(BOOK_ROOT)),
        str(source_path.relative_to(BOOK_ROOT)),
        str(storyboard_path.relative_to(BOOK_ROOT)),
    ],
    "assertions": {
        "standard_R3_form_is_contact": abs(contact_volume_coeff) > 0.99,
        "integrable_comparison_is_not_contact": integrable_volume_coeff == 0.0,
        "bracket_escapes_contact_distribution": abs(bracket_escape) > 0.99,
        "nowhere_zero_scaling_preserves_contact": scaled_volume_min > 0.99,
        "projectivized_cotangent_chart_is_contact": projective_contact_volume_abs > 0.99,
        "gray_family_stays_contact": float(np.min(family_volume_coefficients)) > 0.99,
    },
}
final_path = save_json(final_sanity, ARTIFACT_TOPIC, "checks", "final-sanity.json")
print(json.dumps(final_sanity, indent=2))

In [ ]:
final_sanity = read_json(ARTIFACT_ROOT / "checks" / "final-sanity.json")
assert all(final_sanity["assertions"].values())
for relative in final_sanity["artifacts"]:
    path = BOOK_ROOT / relative
    assert path.exists(), relative
    assert path.stat().st_size > 100, relative

for relative in final_sanity["artifacts"]:
    path = BOOK_ROOT / relative
    if path.suffix.lower() in {".png", ".html", ".svg"}:
        display_artifact(path, width=760, height=430 if path.suffix == ".html" else None)

print("Lecture 10 checks passed; artifacts are present and nonempty.")

## Takeaways

- A contact element is a hyperplane at a point, equivalently the kernel of a nonzero covector up to scalar.
- A contact form is stronger than a one-form defining a plane field: `alpha wedge (d alpha)^n` must be nowhere zero.
- In `R3`, `alpha = x dy + dz` is contact because the planes twist; `ker dz` is the integrable comparison and fails the volume test.
- The contact Darboux normal form says every contact structure locally looks like `sum x_i dy_i + dz` after choosing coordinates.
- Gray's theorem is the compact-family stability result: a smooth family of contact hyperplane fields is carried by an isotopy, with a harmless nowhere-zero scaling of the defining form.